In [16]:
#判別システム全体
#前処理フェーズ，判別フェーズ

#前処理フェーズ：YOLO
#入力：画像フォルダ
#出力：前処理後の画像

#判別フェーズ
#入力：画像フォルダ
#出力：判別結果

In [17]:
import os 
import cv2
import glob
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
import scipy.stats as stats
import seaborn as sns
import pandas as pd
from sklearn.metrics import roc_curve
import re
import math
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,precision_score, recall_score, f1_score
import joblib
import time
from ultralytics import YOLO

In [18]:
start = time.time()

In [19]:
#複数しいたけ画像の場合
# データフォルダの設定
data = "jikuari_maesyori"

# 入力画像
input_file = f"/home/data/{data}/hukusugazou/collage_2.jpg"

# 出力フォルダの作成
mask_output_folder = f"/home/data/{data}/mask/"
crop_output_folder = f"/home/data/{data}/crop/"
combined_output_folder = f"/home/data/{data}/combined/"  # 合成画像の出力フォルダ
os.makedirs(mask_output_folder, exist_ok=True)
os.makedirs(crop_output_folder, exist_ok=True)
os.makedirs(combined_output_folder, exist_ok=True)  # 合成フォルダを作成

# モデルの読み込み（物体検出用）
detection_model = YOLO('/home/YOLO/hukusuu_train/datasets/train7/weights/best.pt')
# モデルの読み込み（セグメンテーション用）
segmentation_model = YOLO("/home/YOLO/-327_seg/datasets/train2/weights/best.pt")

# 画像を処理
def process_image(image_path):
    results = detection_model.predict(image_path)  # 物体検出

    # 画像の読み込み
    orig_img = results[0].orig_img
    img_h, img_w, _ = orig_img.shape

    # 4×6のマスに分割
    rows, cols = 4, 6
    cell_h, cell_w = img_h // rows, img_w // cols

    # マス目ごとの bbox を格納するリスト
    cell_bboxes = [[[] for _ in range(cols)] for _ in range(rows)]

    # bbox を対応するマス目に格納
    if results[0].boxes is not None:
        for i, box in enumerate(results[0].boxes.xyxy):
            start_x, start_y, end_x, end_y = map(int, box)
            center_x, center_y = (start_x + end_x) // 2, (start_y + end_y) // 2
            
            row_idx = min(center_y // cell_h, rows - 1)
            col_idx = min(center_x // cell_w, cols - 1)
            
            cell_bboxes[row_idx][col_idx].append((start_x, start_y, end_x, end_y))

    # 出力フォルダを作成
    base_name = os.path.splitext(os.path.basename(image_path))[0]
    image_output_folder = os.path.join(crop_output_folder, base_name)
    mask_output_folder_specific = os.path.join(mask_output_folder, base_name)
    combined_output_folder_specific = os.path.join(combined_output_folder, base_name)
    os.makedirs(image_output_folder, exist_ok=True)
    os.makedirs(mask_output_folder_specific, exist_ok=True)
    os.makedirs(combined_output_folder_specific, exist_ok=True)  # 合成フォルダを作成

    # マス目順に bbox を保存
    for row in range(rows):
        for col in range(cols):
            if not cell_bboxes[row][col]:  # bboxがない場合はスキップ
                continue
            for idx, (sx, sy, ex, ey) in enumerate(cell_bboxes[row][col]):
                clip_img = orig_img[sy:ey, sx:ex]
                clip_filename = os.path.join(image_output_folder, f"clip_{row+1}_{col+1}.jpg")
                cv2.imwrite(clip_filename, clip_img)

                # セグメンテーション用のマスク作成
                mask_results = segmentation_model.predict(clip_img)
                if mask_results and mask_results[0].masks is not None:
                    mask = mask_results[0].masks.data[0].cpu().numpy()  # CUDA -> CPU に転送して NumPy に変換
                    mask = (mask * 255).astype(np.uint8)  # 0-1 を 0-255 に変換

                    # マスク画像を切り抜き画像のサイズにリサイズ
                    mask_resized = cv2.resize(mask, (clip_img.shape[1], clip_img.shape[0]))

                    # マスク画像を保存
                    mask_filename = os.path.join(mask_output_folder_specific, f"mask_{row+1}_{col+1}.jpg")
                    cv2.imwrite(mask_filename, mask_resized)

                    # マスクと切り抜き画像を掛け合わせる（積を取る）
                    # clip_img_rgb = cv2.cvtColor(clip_img, cv2.COLOR_BGR2RGB)  # RGB形式に変換
                    mask_rgb = cv2.cvtColor(mask_resized, cv2.COLOR_GRAY2BGR)  # グレースケールのマスクをRGBに変換

                    # 切り抜き画像とリサイズしたマスク画像の積
                    combined_img = cv2.bitwise_and(clip_img, mask_rgb)  # 切り抜き画像とマスクの積

                    # 合成画像を保存
                    combined_filename = os.path.join(combined_output_folder_specific, f"combined_{row+1}_{col+1}.jpg")
                    cv2.imwrite(combined_filename, combined_img)

# 画像1枚のみ処理
process_image(input_file)


image 1/1 /home/data/jikuari_maesyori/hukusugazou/collage_2.jpg: 320x640 25 shiitake_bboxs, 20.9ms
Speed: 1.6ms preprocess, 20.9ms inference, 0.3ms postprocess per image at shape (1, 3, 320, 640)

0: 544x640 1 shiitake, 22.3ms
Speed: 0.9ms preprocess, 22.3ms inference, 0.5ms postprocess per image at shape (1, 3, 544, 640)

0: 640x608 1 shiitake, 31.3ms
Speed: 1.1ms preprocess, 31.3ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 608)

0: 640x640 1 shiitake, 24.4ms
Speed: 1.0ms preprocess, 24.4ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 shiitake, 37.6ms
Speed: 1.9ms preprocess, 37.6ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 576x640 1 shiitake, 19.7ms
Speed: 1.1ms preprocess, 19.7ms inference, 0.5ms postprocess per image at shape (1, 3, 576, 640)

0: 576x640 1 shiitake, 20.7ms
Speed: 1.7ms preprocess, 20.7ms inference, 0.6ms postprocess per image at shape (1, 3, 576, 640)

0: 640x448 1 shiitake, 16.3ms
Speed

In [ ]:
import hida
import keijo
import size_module
#判別フェーズ
#特徴量抽出
# hida_tapple = hida.Hida_folder_jikuari(base_dir=f"/home/data/{data}",method="45rotate",n=9,T=0.4,subfolder="collage_2")
hida_tapple = hida.Hida_folder(base_dir=f"/home/data/{data}",n=9,T=0.4,subfolder="collage_2")
result_hida = hida_tapple.run_all()
size_tapple = size_module.Size_folder_taikakusen(base_dir=f"/home/data/{data}",subfolder="collage_2")
result_size = size_tapple.run()
keijo_tapple = keijo.Keijo_folder(base_dir=f"/home/data/{data}",subfolder="collage_2")
result_keijo = keijo_tapple.run()

TypeError: Hida_folder.__init__() got an unexpected keyword argument 'method'

In [ ]:
import pandas as pd
dict_hida =dict(result_hida)
dict_size = dict(result_size)
dict_keijo = dict(result_keijo)
# 結果を結合
merged = []
for filename in dict_hida.keys() & dict_size.keys() & dict_keijo.keys():
    merged.append({
        'filename': filename,
        'MSE': dict_keijo[filename],
        'size_count': dict_size[filename],
        'R': dict_hida[filename]
    })

In [ ]:
df = pd.DataFrame(merged)


In [ ]:

# === モデルとスケーラーの読み込み ===
model = "_jikuari_tai"
model_path = "svm_model.pkl"
scaler_path = "scaler.pkl"
df.to_csv(f'/home/data/{data}/feature{model}.csv', index=False)
svm_model = joblib.load(model_path)
scaler = joblib.load(scaler_path)

# === 新しいデータの読み込み ===
# df = pd.DataFrame(merged)
df = pd.read_csv(f'/home/data/{data}/feature{model}.csv')
df = pd.DataFrame(df)

# === 特徴量の抽出と標準化 ===
X_new = df[["MSE", "size_count", "R"]]  # 学習時と同じ特徴量を使用
X_new = scaler.transform(X_new)  # 標準化

# === 予測 ===
y_pred_new = svm_model.predict(X_new)

# 結果をDataFrameに追加
df["Predicted_Label"] = y_pred_new

# 予測結果の確認
print([["MSE", "size_count", "R", "Predicted_Label"]])

# CSVとして保存（オプション）
df.to_csv(f"/home/data/{data}predicted{model}.csv", index=False)


[['MSE', 'size_count', 'R', 'Predicted_Label']]


In [ ]:
end = time.time()
print(f"Total time: {end - start:.2f} sec")

Total time: 24.30 sec


In [ ]:
# import hida
# a = hida.Hida_file(img_path = "/home/data/maesyori_img/combined/collage_1/combined_1_1.jpg", mask_path = "/home/data/maesyori_img/mask/collage_1/mask_1_1.jpg")
# result = a.calculate_R()
# print(result)

In [ ]:
# import hida
# a = hida.Hida_folder(base_dir="/home/data/maesyori_img")
# result = a.run_all()
# print(result)

In [ ]:
# import size

# a = size.Size_file("/home/data/maesyori_img/mask/collage_1/mask_1_1.jpg")
# result = a.count_white_pixels()
# print(result)

In [ ]:
# import size_module
# import os
# import cv2
# import numpy as np
# b = size_module.Size_folder(base_dir="/home/data/maesyori_img")
# result = b.count_white_pixels()
# print(result)

In [ ]:
# import keijo
# analyzer = keijo.Keijo_file("/home/data/maesyori_img/mask/collage_1/mask_1_1.jpg")
# result = analyzer.run()
# print(result)

In [ ]:
# import keijo
# analyzer = keijo.Keijo_folder(base_dir="/home/data/maesyori_img")
# result = analyzer.run()
# print(result)